# LIBS Foundation Model - Demo Notebook

This notebook demonstrates the key components of the LIBS Foundation Model:

1. Synthetic data generation
2. Model architecture
3. Self-supervised pre-training
4. Fine-tuning for downstream tasks
5. Evaluation and visualization

In [ ]:
import sys
sys.path.append('..')

import numpy as np
import torch
import matplotlib.pyplot as plt

# Set random seeds for reproducibility
np.random.seed(42)
torch.manual_seed(42)

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

## 1. Synthetic Data Generation

We generate LIBS-like spectra with 5 material classes, each with characteristic emission peaks.

In [ ]:
from data.synthetic_generator import SyntheticLIBSGenerator

# Create generator
generator = SyntheticLIBSGenerator(
    n_bins=2048,
    noise_sigma=0.02,
    seed=42,
)

# Display class information
print("Material Classes:")
for cid, info in generator.get_class_info().items():
    print(f"  {cid}: {info['name']} - peaks at bins {info['peak_positions']}")

In [ ]:
# Generate example spectra for each class
fig, axes = plt.subplots(5, 1, figsize=(14, 12))

class_names = ['Iron', 'Copper', 'Aluminum', 'Calcium', 'Mixed']

for class_id in range(5):
    spectrum, _, _ = generator.generate_spectrum(class_id=class_id)
    axes[class_id].plot(spectrum)
    axes[class_id].set_title(f'Class {class_id}: {class_names[class_id]}')
    axes[class_id].set_ylabel('Intensity')

axes[-1].set_xlabel('Bin Index')
plt.tight_layout()
plt.show()

In [ ]:
# Generate a dataset
spectra, labels, concentrations = generator.generate_dataset(
    n_samples=1000,
    mixed_ratio=0.2,  # 20% mixed samples
)

print(f"Spectra shape: {spectra.shape}")
print(f"Labels shape: {labels.shape}")
print(f"Concentrations shape: {concentrations.shape}")
print(f"\nLabel distribution: {np.bincount(labels)}")

## 2. Model Architecture

The LIBS Transformer is a BERT-style model adapted for spectral data.

In [ ]:
from models.libs_transformer import LIBSTransformer

# Create model
model = LIBSTransformer(
    n_bins=2048,
    d_model=256,
    n_heads=8,
    n_layers=6,
    d_ff=1024,
    dropout=0.1,
    n_classes=5,
)

print(f"Model parameters: {model.num_parameters:,}")

In [ ]:
# Test forward pass
batch = torch.tensor(spectra[:4], dtype=torch.float32)
output = model(batch)

print("Forward pass outputs:")
print(f"  CLS embedding: {output['cls_embedding'].shape}")
print(f"  Sequence output: {output['sequence_output'].shape}")
print(f"  MIP predictions: {output['mip_predictions'].shape}")

## 3. Masked Intensity Prediction

The self-supervised objective: predict masked intensity values.

In [ ]:
from data.dataset import MaskedLIBSDataset

# Create masked dataset
masked_dataset = MaskedLIBSDataset(
    spectra=spectra,
    mask_ratio=0.15,
    seed=42,
)

# Get a sample
sample = masked_dataset[0]
print(f"Masked input shape: {sample['input'].shape}")
print(f"Target shape: {sample['target'].shape}")
print(f"Masked positions: {sample['mask'].sum().item()} / {len(sample['mask'])}")

In [ ]:
# Visualize masking
fig, axes = plt.subplots(2, 1, figsize=(14, 6))

axes[0].plot(sample['target'].numpy(), label='Original', alpha=0.8)
axes[0].plot(sample['input'].numpy(), label='Masked Input', alpha=0.8)
axes[0].set_title('Original vs Masked Spectrum')
axes[0].legend()

# Highlight masked regions
mask_indices = sample['mask'].numpy()
axes[1].fill_between(range(len(mask_indices)), 0, mask_indices, alpha=0.5)
axes[1].set_title('Masked Positions (15% of bins)')
axes[1].set_xlabel('Bin Index')

plt.tight_layout()
plt.show()

In [ ]:
# Test MIP loss computation
batch_input = sample['input'].unsqueeze(0)
batch_mask = sample['mask'].unsqueeze(0)
batch_target = sample['target'].unsqueeze(0)

output = model(batch_input, mask=batch_mask)
loss = model.compute_mip_loss(
    output['mip_predictions'],
    batch_target,
    batch_mask,
)

print(f"MIP Loss: {loss.item():.4f}")

## 4. Fine-tuning Heads

Classification and regression heads for downstream tasks.

In [ ]:
from models.heads import ClassificationHead, RegressionHead

# Get CLS embeddings
batch = torch.tensor(spectra[:32], dtype=torch.float32)
with torch.no_grad():
    output = model(batch)
    cls_embeddings = output['cls_embedding']

print(f"CLS embeddings shape: {cls_embeddings.shape}")

# Classification head
cls_head = ClassificationHead(d_model=256, n_classes=5)
logits = cls_head(cls_embeddings)
print(f"Classification logits shape: {logits.shape}")

# Regression head
reg_head = RegressionHead(d_model=256, n_outputs=5)
conc_preds = reg_head(cls_embeddings)
print(f"Concentration predictions shape: {conc_preds.shape}")
print(f"Concentration range: [{conc_preds.min():.3f}, {conc_preds.max():.3f}]")

## 5. Evaluation Metrics and Visualization

In [ ]:
from utils.metrics import (
    compute_classification_metrics,
    compute_regression_metrics,
    format_metrics,
    plot_confusion_matrix,
)

# Simulate classification results
with torch.no_grad():
    pred_labels = logits.argmax(dim=-1).numpy()
true_labels = labels[:32]

cls_metrics = compute_classification_metrics(
    pred_labels,
    true_labels,
    class_names=['Iron', 'Copper', 'Aluminum', 'Calcium', 'Mixed'],
)

print("Classification Metrics (untrained model):")
print(format_metrics(cls_metrics))

In [ ]:
# Plot confusion matrix
fig, ax = plt.subplots(figsize=(8, 6))
plot_confusion_matrix(
    cls_metrics['confusion_matrix'],
    class_names=['Fe', 'Cu', 'Al', 'Ca', 'Mix'],
    normalize=True,
    ax=ax,
)
plt.tight_layout()
plt.show()

In [ ]:
# Regression metrics
with torch.no_grad():
    conc_preds_np = conc_preds.numpy()
true_conc = concentrations[:32]

reg_metrics = compute_regression_metrics(
    conc_preds_np,
    true_conc,
    output_names=['Fe', 'Cu', 'Al', 'Ca', 'Mix'],
)

print("Regression Metrics (untrained model):")
print(format_metrics(reg_metrics))

## 6. Training Example (Mini-batch)

Quick demonstration of a single training step.

In [ ]:
from training.pretrain import LIBSPretrainModule
from torch.utils.data import DataLoader

# Create dataloader
train_loader = DataLoader(masked_dataset, batch_size=16, shuffle=True)

# Create Lightning module
pretrain_module = LIBSPretrainModule(
    model=model,
    learning_rate=1e-4,
    warmup_epochs=5,
    max_epochs=50,
)

# Get a batch
batch = next(iter(train_loader))

# Training step
pretrain_module.train()
loss = pretrain_module.training_step(batch, 0)
print(f"Training loss: {loss.item():.4f}")

# Validation step
pretrain_module.eval()
with torch.no_grad():
    val_result = pretrain_module.validation_step(batch, 0)
print(f"Validation loss: {val_result['loss'].item():.4f}")

## Summary

This notebook demonstrated the core components of the LIBS Foundation Model:

- **Synthetic data generation** with 5 material classes and realistic spectral characteristics
- **Transformer architecture** with spectral embedding and positional encoding
- **Masked intensity prediction** for self-supervised pre-training
- **Task-specific heads** for classification and regression
- **Evaluation metrics** and visualization utilities

For full training, use the provided scripts:
```bash
# Pre-training
python train_pretrain.py --config config/config.yaml

# Fine-tuning
python train_finetune.py --pretrained_checkpoint checkpoints/pretrain/final_model.pt --task both
```